In [ ]:
#@title Protein Sequence Mapping { display-mode: "form" }
from IPython.display import display, HTML
display(HTML("""
<div style='font-family:sans-serif;'>
  <div style='display:flex;justify-content:space-between;align-items:flex-start;margin-bottom:1.5rem;'>
    <div style='flex:1;'>
      <h1 style='margin:0 0 0.5rem 0;font-size:2rem;font-weight:700;'>Protein Sequence Mapping</h1>
      <p style='margin:0;color:#444;font-size:1rem;'>Maps metabolic model protein sequences to a genome annotation via BLASTp, outputting a protein ID mapping CSV.</p>
    </div>
    <div style='margin-left:2rem;flex-shrink:0;'>
      <img src='https://raw.githubusercontent.com/MeMoModelling/proteinSeqMapping/main/bii_horizontal_logo_smalle472014210204e8886d5cf09c653e7e5.png' style='width:190px;'/>
    </div>
  </div>
  <hr style='border:none;border-top:1px solid #ddd;margin:1.5rem 0;'/>
  <h2 style='font-size:1.4rem;margin:0 0 0.8rem 0;'>Instructions:</h2>
  <ol style='margin:0 0 1.5rem 0;padding-left:1.5rem;font-size:1rem;line-height:1.8;'>
    <li>Go to <strong>Runtime &#8594; Run all</strong> in the top menu.</li>
    <li>Upload your files when prompted.</li>
    <li>Download the output CSV from the <strong>Files panel</strong> on the left.</li>
  </ol>
  <h3 style='font-size:1.1rem;margin:0 0 0.5rem 0;'>Inputs:</h3>
  <ul style='margin:0 0 1rem 0;padding-left:1.5rem;font-size:1rem;line-height:1.8;'>
    <li>Protein FASTA file to query &nbsp;<code>e.g. modelname_orf_proteins.faa</code></li>
    <li>Protein FASTA file as the database &nbsp;<code>e.g. modelname_proteins.faa</code></li>
  </ul>
  <h3 style='font-size:1.1rem;margin:0 0 0.5rem 0;'>Outputs:</h3>
  <ul style='margin:0;padding-left:1.5rem;font-size:1rem;line-height:1.8;'>
    <li>Protein ID mapping from query to best-matched database entry &nbsp;<code>e.g. modelname_protein_id_mapping.csv</code></li>
  </ul>
</div>
"""))

In [ ]:
#@title Step 1 — Set up environment { display-mode: "form" }
import shutil, subprocess
from IPython.display import display, HTML

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;'>&#9881; Setting up environment...</b>
  <p style='margin:0.3rem 0 0 0;font-size:0.88rem;color:#555;'>This runs automatically and takes about 1 minute.</p>
</div>
"""))

if not shutil.which('blastp'):
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ncbi-blast+'], check=True, capture_output=True)
subprocess.run(['pip', 'install', '-q', 'pandas'], capture_output=True)

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0fff8;border-left:4px solid #006d5b;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#006d5b;'>&#10003; Ready! Proceed to the next step.</b>
</div>
"""))

In [ ]:
#@title Step 2 — Upload files & run { display-mode: "form" }
from IPython.display import display, HTML
from google.colab import files
import os, subprocess, pandas as pd

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:1rem;'>&#128193; Upload your model protein sequences</b>
  <p style='margin:0.3rem 0 0 0;font-size:0.88rem;color:#555;'>The protein FASTA file from your metabolic model <code>(.faa)</code></p>
</div>
"""))
uploaded_query = files.upload()
query_filename_raw = list(uploaded_query.keys())[0]
query_filename = query_filename_raw.replace(' ', '_')
if query_filename != query_filename_raw:
    os.rename(query_filename_raw, query_filename)
display(HTML(f"<p style='font-family:sans-serif;color:#006d5b;'>&#10003; Uploaded: <code>{query_filename}</code></p>"))

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f8f9fb;border:1.5px solid #d0d7e8;border-radius:8px;font-family:sans-serif;margin-bottom:0.8rem;'>
  <b style='font-size:1rem;'>&#128193; Upload your genome protein sequences</b>
  <p style='margin:0.3rem 0 0 0;font-size:0.88rem;color:#555;'>The protein FASTA file downloaded from NCBI <code>(.faa)</code></p>
</div>
"""))
uploaded_db = files.upload()
db_filename_raw = list(uploaded_db.keys())[0]
db_filename = db_filename_raw.replace(' ', '_')
if db_filename != db_filename_raw:
    os.rename(db_filename_raw, db_filename)
display(HTML(f"<p style='font-family:sans-serif;color:#006d5b;'>&#10003; Uploaded: <code>{db_filename}</code></p>"))

species_name = query_filename.split('_')[0]
db_name    = f'{species_name}_db'
blast_out  = f'{species_name}_blast.txt'
output_csv = f'{species_name}_protein_id_mapping.csv'

display(HTML("""
<div style='padding:0.8rem 1.2rem;background:#f0f4ff;border-left:4px solid #1a56db;border-radius:0 8px 8px 0;font-family:sans-serif;'>
  <b style='color:#1a56db;'>&#8635; Processing... this may take a few minutes.</b>
</div>
"""))

subprocess.run(['makeblastdb', '-in', db_filename, '-dbtype', 'prot', '-out', db_name],
               check=True, capture_output=True)
subprocess.run([
    'blastp', '-query', query_filename, '-db', db_name, '-out', blast_out,
    '-outfmt', '6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs',
    '-num_threads', '2'
], check=True, capture_output=True)

cols = ['qseqid','sseqid','pident','length','mismatch','gapopen','qstart','qend','sstart','send','evalue','bitscore','qcovs']
blast_df = pd.read_csv(blast_out, sep='\t', header=None, names=cols)

def seq_lengths(fasta):
    lengths, cur_id, cur_len = {}, None, 0
    with open(fasta) as f:
        for line in f:
            if line.startswith('>'):
                if cur_id: lengths[cur_id] = cur_len
                cur_id, cur_len = line[1:].strip().split()[0], 0
            else:
                cur_len += len(line.strip())
    if cur_id: lengths[cur_id] = cur_len
    return lengths

blast_df['qlen'] = blast_df['qseqid'].map(seq_lengths(query_filename))
blast_df['slen'] = blast_df['sseqid'].map(seq_lengths(db_filename))
blast_df['qcov'] = (blast_df['qend'] - blast_df['qstart'] + 1) / blast_df['qlen']
blast_df['scov'] = blast_df['length'] / blast_df['slen']

filtered = blast_df[
    (blast_df['pident']   >= 95)   &
    (blast_df['gapopen']  == 0)    &
    (blast_df['mismatch'] <= 1)    &
    (blast_df['qcov']     >= 0.85) &
    (blast_df['scov']     >= 0.85)
]

best   = filtered.sort_values('evalue').groupby('qseqid').first().reset_index()
result = best[['qseqid','sseqid']].rename(columns={'qseqid':'identifier_query','sseqid':'identifier_db'})
result.to_csv(output_csv, index=False)

display(HTML(f"""
<div style='padding:1rem 1.2rem;background:#f0fff8;border:1.5px solid #006d5b;border-radius:8px;font-family:sans-serif;margin-top:0.5rem;'>
  <b style='font-size:1rem;color:#006d5b;'>&#10003; Done!</b>
  <p style='margin:0.4rem 0 0 0;font-size:0.9rem;color:#333;'>{len(result)} matches found out of {blast_df['qseqid'].nunique()} sequences.</p>
  <p style='margin:0.4rem 0 0 0;font-size:0.9rem;color:#333;'>Output saved as <code>{output_csv}</code></p>
  <p style='margin:0.6rem 0 0 0;font-size:0.88rem;color:#555;'>&#128193; Go to the <b>Files panel on the left</b> &#8594; right-click <code>{output_csv}</code> &#8594; <b>Download</b></p>
</div>
"""))